# Modelado estocástico y aprendizaje no supervisado en dinámicas de desaparición en Ecuador
Trabajo Fin de Máster · Máster en Ingeniería Matemática y Computación (UNIR) · Carlos Andrés Mendoza Bravo

Cuaderno reproducible con el análisis completo de la memoria. Datos: conjunto de datos abiertos *Personas desaparecidas* del Ministerio del Interior de Ecuador (2017-2025).

Contenido:
1. Carga y revisión de calidad de los datos.
2. Preparación: definición del evento (cierre del caso), de la censura y de la fecha de corte.
3. Análisis de supervivencia: Kaplan-Meier, riesgos competitivos (Aalen-Johansen), log-rank, Cox estratificado por año, RMST.
4. Agrupamiento espacial DBSCAN (distancia de Haversine) en Guayas: selección de parámetros, severidad estandarizada por año con corrección FDR y validación temporal.
5. Reglas de asociación (Apriori) perfil → desenlace con filtros de robustez y validación temporal.
6. Integración de zonas y perfiles en un modelo de Cox con datos posteriores.

In [1]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg") if os.environ.get("TFM_HEADLESS") else None
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

try:
    import lifelines, mlxtend, statsmodels
except ImportError:
    %pip install -q lifelines mlxtend statsmodels
    import lifelines, mlxtend, statsmodels

from lifelines import KaplanMeierFitter, CoxPHFitter, AalenJohansenFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test, proportional_hazard_test
from lifelines.utils import median_survival_times, restricted_mean_survival_time
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors, BallTree
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
from mlxtend.frequent_patterns import apriori, association_rules

RUTA_XLSX = os.environ.get("TFM_XLSX", "/content/mdi_personasdesaparecidas_pm_2017_2025.xlsx")
SEMILLA   = 42
KM_TIERRA = 6371.0088
os.makedirs("figuras", exist_ok=True); os.makedirs("tablas", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "font.size": 10})
RES = {}
def mostrar(x):
    try: display(x)
    except NameError: print(x)

## 1. Carga y revisión de calidad de los datos

In [2]:
raw = pd.read_excel(RUTA_XLSX)
print(f"Registros: {raw.shape[0]:,} | Variables: {raw.shape[1]}")
RES["n_registros"] = int(raw.shape[0])
aud = pd.DataFrame({"tipo": raw.dtypes.astype(str), "nulos": raw.isnull().sum(), "% nulos": (raw.isnull().sum()/len(raw)*100).round(2)})
mostrar(aud)
print("\nsituacion_actual:\n", raw["situacion_actual"].value_counts())

Registros: 75,749 | Variables: 26


,tipo,nulos,% nulos
fecha_desaparicion,datetime64[ns],0,0.00
fecha_denuncia,datetime64[ns],0,0.00
fecha_conocimiento,datetime64[ns],68220,90.06
zona,object,0,0.00
distrito,object,0,0.00
circuito,object,0,0.00
subcircuito,object,0,0.00
codigo_provincia,int64,0,0.00
provincia,object,0,0.00
codigo_canton,int64,0,0.00



situacion_actual:
 situacion_actual
ENCONTRADO      70616
FALLECIDO        2603
DESAPARECIDO     2530
Name: count, dtype: int64


In [3]:
# Hallazgo 1: los 'FALLECIDO' tienen fecha de localización (se hallaron) -> son un EVENTO, no censura
raw["f_loc"] = pd.to_datetime(raw["fecha_localizacion"].replace({"NO_APLICA": np.nan, "SIN_DATO": np.nan}), errors="coerce", format="mixed")
mostrar(pd.crosstab(raw["situacion_actual"], raw["f_loc"].notna(), rownames=["situación"], colnames=["tiene fecha de localización"]))

# Hallazgo 2: corte real de la base
FECHA_CORTE_ORIG = raw["fecha_desaparicion"].max()
FECHA_CORTE      = max(raw["f_loc"].max(), raw["fecha_denuncia"].max(), raw["fecha_desaparicion"].max())
print(f"\nÚltima fecha de desaparición: {FECHA_CORTE_ORIG.date()}  |  corte observado en la base: {FECHA_CORTE.date()}")
RES["fecha_corte"] = str(FECHA_CORTE.date())

# Hallazgo 3: 'motivo' y 'estado' son jerárquicos (EXTRAVIADO-* ⊂ EXTRAVÍO)
mot_extr = raw["motivo_desaparicion"].where(raw["motivo_desaparicion"].str.startswith("EXTRAV"), "otros motivos")
mostrar(pd.crosstab(mot_extr, raw["estado_desaparecido"]))

tiene fecha de localización,False,True
situación,,
DESAPARECIDO,2530,0
ENCONTRADO,1,70615
FALLECIDO,0,2603



Última fecha de desaparición: 2025-12-31  |  corte observado en la base: 2026-08-12


estado_desaparecido,DESAPARICIÓN INVOLUNTARIA,DESAPARICIÓN VOLUNTARIA,EN INVESTIGACIÓN,EXTRAVÍO
motivo_desaparicion,,,,
EXTRAVIADO - AUSENCIA TEMPORAL,0,0,0,2104
EXTRAVIADO - DISCAPACIDAD / ENFERMEDADES / TRASTORNOS,0,0,0,4216
otros motivos,4372,62527,2530,0


## 2. Preprocesamiento

In [4]:
df = raw.copy()
sin_fecha = (df["situacion_actual"] == "ENCONTRADO") & df["f_loc"].isna()
print("Registros 'ENCONTRADO' sin fecha de localización (excluidos):", int(sin_fecha.sum()))
df = df[~sin_fecha].copy()

# Evento = cierre del caso (localizado con vida o fallecido). Censura = sigue desaparecido al corte.
df["evento"] = (df["situacion_actual"] != "DESAPARECIDO").astype(int)
df["t"] = np.where(df["evento"] == 1, (df["f_loc"] - df["fecha_desaparicion"]).dt.days, (FECHA_CORTE - df["fecha_desaparicion"]).dt.days)
assert (df["t"] >= 0).all()
df["causa"] = np.select([df["situacion_actual"]=="ENCONTRADO", df["situacion_actual"]=="FALLECIDO"], [1, 2], 0)
df["anio"]  = df["fecha_desaparicion"].dt.year

num = lambda s: pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")
df["lat"], df["lon"] = num(df["latitud_desaparicion"]), num(df["longitud_desaparicion"])
en_rango = df["lat"].between(-5.1, 1.8) & df["lon"].between(-92.5, -75.0)
print("Coordenadas nulas:", int(df[["lat","lon"]].isna().any(axis=1).sum()), "| fuera de rango:", int((~en_rango).sum()))
df = df[en_rango].copy()
dup = df.groupby(["lat","lon"]).size()
print(f"Coordenadas únicas: {len(dup):,} de {len(df):,} casos; {dup[dup>=5].sum()/len(df)*100:.1f}% de los casos comparte coordenada con ≥4 más.")

costa = {"GUAYAS","MANABÍ","LOS RÍOS","EL ORO","ESMERALDAS","SANTA ELENA","STO DGO DE LOS TSÁCHILAS"}
amaz  = {"MORONA SANTIAGO","NAPO","ORELLANA","PASTAZA","SUCUMBÍOS","ZAMORA CHINCHIPE"}
df["region"] = np.where(df["provincia"].isin(costa), "COSTA", np.where(df["provincia"].isin(amaz), "AMAZONIA", np.where(df["provincia"]=="GALÁPAGOS", "INSULAR", "SIERRA")))
df["edad_g"] = df["rango_edad"].where(df["rango_edad"] != "SIN_DATO")
print(f"\nBase analítica: {len(df):,} casos | eventos: {int(df.evento.sum()):,} | censurados: {int((df.evento==0).sum()):,}")
RES["n_analitico"] = int(len(df)); RES["n_censurados"] = int((df.evento==0).sum())

Registros 'ENCONTRADO' sin fecha de localización (excluidos): 1


Coordenadas nulas: 0 | fuera de rango: 0


Coordenadas únicas: 54,798 de 75,748 casos; 20.8% de los casos comparte coordenada con ≥4 más.

Base analítica: 75,748 casos | eventos: 73,218 | censurados: 2,530


## 3. Modelo estocástico de supervivencia (Kaplan-Meier)
### 3.1 Efecto de la corrección de la censura

In [5]:
def km_resumen(t, e, etiqueta):
    k = KaplanMeierFitter().fit(t, e)
    lo, hi = median_survival_times(k.confidence_interval_).values[0]
    return k, dict(modelo=etiqueta, n=len(t), eventos=int(e.sum()), censurados=int((e==0).sum()), mediana=k.median_survival_time_,
                   mediana_IC95=f"[{lo:.0f}; {hi:.0f}]", **{f"S({d})": round(float(k.predict(d)), 4) for d in (7, 30, 365)})

o = df.copy()
o["ev_o"] = (o["situacion_actual"]=="ENCONTRADO").astype(int)
o["t_o"]  = np.where(o["ev_o"]==1, (o["f_loc"]-o["fecha_desaparicion"]).dt.days, (FECHA_CORTE_ORIG-o["fecha_desaparicion"]).dt.days)
o = o[o["t_o"]>=0]
k_orig, r1 = km_resumen(o["t_o"], o["ev_o"], "Alternativa (fallecidos censurados; corte 31-dic-2025)")
k_corr, r2 = km_resumen(df["t"], df["evento"], "Principal (evento = cierre; corte 12-ago-2026)")
tab = pd.DataFrame([r1, r2]); mostrar(tab); tab.to_csv("tablas/km_original_vs_corregida.csv", index=False)
print("Fallecidos tratados como 'siguen desaparecidos' en la definición alternativa:", int((o.situacion_actual=='FALLECIDO').sum()),
      "| tiempo medio imputado:", round(o.loc[o.situacion_actual=='FALLECIDO','t_o'].mean()), "días")
RES["km"] = dict(original=r1, corregida=r2)

fig, ax = plt.subplots(figsize=(9, 5))
k_corr.plot_survival_function(ax=ax, ci_show=True, color="#1f77b4", label="Evento = cierre")
k_orig.plot_survival_function(ax=ax, ci_show=False, color="#d62728", ls="--", label="Fallecidos censurados")
ax.axhline(.5, color="gray", ls=":"); ax.set_xlim(0, 365)
ax.set(title="Probabilidad de que el caso siga abierto, S(t)", xlabel="Días desde la desaparición", ylabel="S(t)")
ax.legend(); fig.tight_layout(); fig.savefig("figuras/fig_km_general.png"); plt.show()

,modelo,n,eventos,censurados,mediana,mediana_IC95,S(7),S(30),S(365)
0,Alternativa (fallecidos censurados; corte 31-d...,75748,70615,5133,7.0,[7; 8],0.4967,0.2431,0.0837
1,Principal (evento = cierre; corte 12-ago-2026),75748,73218,2530,7.0,[7; 7],0.4795,0.2174,0.0548


Fallecidos tratados como 'siguen desaparecidos' en la definición alternativa: 2603 | tiempo medio imputado: 1290 días


### 3.2 Riesgos competitivos: localizado con vida vs. localizado fallecido

In [6]:
cif = {}
for c, nm in [(1, "Localizado con vida"), (2, "Localizado fallecido")]:
    aj = AalenJohansenFitter(calculate_variance=False, seed=SEMILLA).fit(df["t"], df["causa"], event_of_interest=c)
    cif[nm] = aj.cumulative_density_.iloc[:, 0]
fig, ax = plt.subplots(figsize=(9, 5))
for nm, s in cif.items(): ax.step(s.index, s.values, where="post", label=nm)
ax.set(xlim=(0, 365), title="Incidencia acumulada por tipo de cierre (Aalen-Johansen)", xlabel="Días", ylabel="Probabilidad acumulada")
ax.legend(); fig.tight_layout(); fig.savefig("figuras/fig_cif.png"); plt.show()
at = lambda s, d: float(s.loc[:d].iloc[-1])
RES["cif"] = {nm: {f"{d}d": round(at(s, d), 4) for d in (7, 30, 365)} for nm, s in cif.items()}
mostrar(pd.DataFrame(RES["cif"]).T)

,7d,30d,365d
Localizado con vida,0.4866,0.7542,0.9121
Localizado fallecido,0.0167,0.0261,0.0331


### 3.3 Sensibilidad al origen del tiempo (desaparición vs. denuncia)

In [7]:
s = df[df["fecha_denuncia"] >= df["fecha_desaparicion"]].copy()
s["t_d"] = np.where(s["evento"]==1, (s["f_loc"]-s["fecha_denuncia"]).dt.days, (FECHA_CORTE-s["fecha_denuncia"]).dt.days)
s = s[s["t_d"]>=0]
k_den, r3 = km_resumen(s["t_d"], s["evento"], "Origen = fecha de denuncia")
mostrar(pd.DataFrame([r2, r3])[["modelo","n","mediana","S(7)","S(30)","S(365)"]])
print("Casos con denuncia anterior a la desaparición (excluidos en esta sensibilidad):", int((df.fecha_denuncia < df.fecha_desaparicion).sum()))
RES["km_origen_denuncia"] = r3

,modelo,n,mediana,S(7),S(30),S(365)
0,Principal (evento = cierre; corte 12-ago-2026),75748,7.0,0.4795,0.2174,0.0548
1,Origen = fecha de denuncia,75710,4.0,0.3693,0.1750,0.0502


Casos con denuncia anterior a la desaparición (excluidos en esta sensibilidad): 14


### 3.4 La resolución se ha vuelto más lenta: tendencia 2017-2025
Antes de comparar grupos hay que comprobar que la población es estable en el tiempo. No lo es.

In [8]:
filas = []
for a in range(2017, 2026):
    s_ = df[df.anio == a]; k = KaplanMeierFitter().fit(s_.t, s_.evento)
    filas.append(dict(anio=a, n=len(s_), mediana=k.median_survival_time_, **{"S(30)": round(float(k.predict(30)),3)},
                      pct_fallecido=round((s_.situacion_actual=="FALLECIDO").mean()*100,1), pct_sigue_desaparecido=round((s_.situacion_actual=="DESAPARECIDO").mean()*100,1)))
tend = pd.DataFrame(filas); mostrar(tend); tend.to_csv("tablas/tendencia_anual.csv", index=False)
RES["tendencia_anual"] = tend.to_dict("records")

fig, ax = plt.subplots(figsize=(9, 5))
cmap = plt.cm.viridis(np.linspace(0, .9, 9))
for a, c in zip(range(2017, 2026), cmap):
    s_ = df[df.anio == a]; KaplanMeierFitter().fit(s_.t, s_.evento, label=str(a)).plot_survival_function(ax=ax, ci_show=False, color=c)
ax.set(title="S(t) por año de desaparición", xlim=(0, 120), xlabel="Días", ylabel="S(t)"); ax.legend(ncol=3, fontsize=8)
fig.tight_layout(); fig.savefig("figuras/fig_km_por_anio.png"); plt.show()

,anio,n,mediana,S(30),pct_fallecido,pct_sigue_desaparecido
0,2017,10479,6.0,0.189,1.7,0.8
1,2018,10274,6.0,0.169,1.8,0.9
2,2019,9981,5.0,0.181,1.9,0.9
3,2020,6776,5.0,0.172,2.5,1.9
4,2021,7981,7.0,0.187,2.8,2.0
5,2022,7759,8.0,0.236,4.1,3.5
6,2023,7862,10.0,0.285,5.8,5.4
7,2024,7107,10.0,0.296,6.2,7.1
8,2025,7529,9.0,0.281,5.9,10.2


### 3.5 Comparación por sexo y edad: significancia **y** tamaño de efecto
Con n ≈ 75 000 el Log-Rank detecta diferencias mínimas. Se reporta además el *hazard ratio* (Cox) crudo, ajustado por edad y **estratificado por año**, que es la comparación comparable en el tiempo.

In [9]:
d2 = df[df["edad_g"].notna()].copy()
def km_grupo(col, valores):
    out = []
    for v in valores:
        m = d2[col] == v; k = KaplanMeierFitter().fit(d2.loc[m,"t"], d2.loc[m,"evento"])
        out.append(dict(grupo=v, n=int(m.sum()), mediana=k.median_survival_time_, **{"S(30)": round(float(k.predict(30)),3)}, RMST_30d=round(restricted_mean_survival_time(k, t=30), 2)))
    return pd.DataFrame(out)
tab_sexo = km_grupo("sexo", ["HOMBRE", "MUJER"]); tab_edad = km_grupo("edad_g", ["NIÑOS(AS)", "ADOLESCENTES", "ADULTO", "ADULTO MAYOR"])
mostrar(tab_sexo); mostrar(tab_edad); tab_sexo.to_csv("tablas/km_por_sexo.csv", index=False); tab_edad.to_csv("tablas/km_por_edad.csv", index=False)

h, m_ = d2[d2.sexo=="HOMBRE"], d2[d2.sexo=="MUJER"]
lr = logrank_test(h.t, m_.t, h.evento, m_.evento); lre = multivariate_logrank_test(d2.t, d2.edad_g, d2.evento)
print(f"Log-Rank sexo: χ²={lr.test_statistic:.1f}, p={lr.p_value:.2e} | Log-Rank edad: χ²={lre.test_statistic:.1f}, p<0,001")

X = d2[["t","evento"]].copy(); X["mujer"] = (d2.sexo=="MUJER").astype(int)
for g_ in ["NIÑOS(AS)","ADULTO","ADULTO MAYOR"]: X[g_] = (d2.edad_g==g_).astype(int)     # ref.: ADOLESCENTES
cox_crudo = CoxPHFitter().fit(X[["t","evento","mujer"]], "t", "evento")
cox_aj    = CoxPHFitter().fit(X, "t", "evento")
Xa = X.copy(); Xa["anio"] = d2["anio"].values
cox_anio  = CoxPHFitter().fit(Xa, "t", "evento", strata=["anio"])
f = lambda c: c.summary.loc["mujer", ["exp(coef)","exp(coef) lower 95%","exp(coef) upper 95%"]].round(3).tolist()
hr = pd.DataFrame({"modelo": ["Mujer vs. hombre (crudo)", "Mujer vs. hombre (ajustado por edad)", "Mujer vs. hombre (ajustado por edad, estratificado por año)"],
                   "HR (IC95%)": [f"{a[0]} [{a[1]}; {a[2]}]" for a in (f(cox_crudo), f(cox_aj), f(cox_anio))]})
mostrar(hr); hr.to_csv("tablas/cox_sexo_edad.csv", index=False)
mostrar(cox_anio.summary[["exp(coef)","exp(coef) lower 95%","exp(coef) upper 95%","p"]].round(3))
RES["cox_sexo"] = dict(crudo=f(cox_crudo), ajustado_edad=f(cox_aj), ajustado_edad_estrat_anio=f(cox_anio))
RES["cox_edad_estrat_anio"] = cox_anio.summary[["exp(coef)","exp(coef) lower 95%","exp(coef) upper 95%","p"]].round(3).reset_index().to_dict("records")

filas = []
for g_ in ["NIÑOS(AS)","ADOLESCENTES","ADULTO","ADULTO MAYOR"]:
    sub = d2[d2.edad_g==g_]; a, b = sub[sub.sexo=="HOMBRE"], sub[sub.sexo=="MUJER"]
    filas.append(dict(grupo=g_, p_logrank=logrank_test(a.t,b.t,a.evento,b.evento).p_value))
mostrar(pd.DataFrame(filas).round(4))
ph = proportional_hazard_test(cox_aj, X, time_transform="rank"); print("\nPrueba de riesgos proporcionales (Schoenfeld), modelo ajustado por edad:"); mostrar(ph.summary.round(4))

,grupo,n,mediana,S(30),RMST_30d
0,HOMBRE,28433,7.0,0.244,12.56
1,MUJER,47310,7.0,0.202,11.82


,grupo,n,mediana,S(30),RMST_30d
0,NIÑOS(AS),4634,9.0,0.257,13.15
1,ADOLESCENTES,38096,6.0,0.179,11.03
2,ADULTO,30029,8.0,0.261,13.41
3,ADULTO MAYOR,2984,5.0,0.207,10.87


Log-Rank sexo: χ²=536.0, p=1.41e-118 | Log-Rank edad: χ²=1449.1, p<0,001


,modelo,HR (IC95%)
0,Mujer vs. hombre (crudo),1.191 [1.173; 1.209]
1,Mujer vs. hombre (ajustado por edad),1.075 [1.057; 1.093]
2,"Mujer vs. hombre (ajustado por edad, estratifi...",1.044 [1.026; 1.061]


,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,p
covariate,,,,
mujer,1.044,1.026,1.061,0.0
NIÑOS(AS),0.845,0.819,0.872,0.0
ADULTO,0.762,0.749,0.775,0.0
ADULTO MAYOR,0.887,0.852,0.923,0.0


,grupo,p_logrank
0,NIÑOS(AS),0.0262
1,ADOLESCENTES,0.0552
2,ADULTO,0.0000
3,ADULTO MAYOR,0.0006



Prueba de riesgos proporcionales (Schoenfeld), modelo ajustado por edad:


,test_statistic,p,-log2(p)
ADULTO,0.0991,0.7529,0.4094
ADULTO MAYOR,241.6035,0.0000,178.5700
NIÑOS(AS),2.5758,0.1085,3.2041
mujer,380.0164,0.0000,278.7384


In [10]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for v, c in [("HOMBRE","#1f77b4"), ("MUJER","#e377c2")]:
    m = d2.sexo == v; KaplanMeierFitter().fit(d2.t[m], d2.evento[m], label=v).plot_survival_function(ax=axs[0], color=c)
axs[0].set(title="Por sexo", xlim=(0, 180), xlabel="Días", ylabel="S(t)")
for v, c in [("NIÑOS(AS)","#2ca02c"),("ADOLESCENTES","#ff7f0e"),("ADULTO","#1f77b4"),("ADULTO MAYOR","#9467bd")]:
    m = d2.edad_g == v; KaplanMeierFitter().fit(d2.t[m], d2.evento[m], label=v).plot_survival_function(ax=axs[1], color=c, ci_show=False)
axs[1].set(title="Por grupo de edad", xlim=(0, 180), xlabel="Días")
fig.suptitle("Probabilidad de que el caso siga abierto", y=1.02); fig.tight_layout(); fig.savefig("figuras/fig_km_sexo_edad.png", bbox_inches="tight"); plt.show()

## 4. Agrupamiento espacial basado en densidad (DBSCAN) — provincia piloto: Guayas
### 4.1 Selección de parámetros

In [11]:
g = df[df.provincia == "GUAYAS"].copy().reset_index(drop=True)
Xg = np.radians(g[["lat","lon"]].to_numpy())
print(f"Guayas: {len(g):,} casos | coordenadas únicas: {len(g[['lat','lon']].drop_duplicates()):,}")
fig, ax = plt.subplots(figsize=(8, 4.5))
for k in (15, 30, 50):
    d, _ = NearestNeighbors(n_neighbors=k+1, metric="haversine", algorithm="ball_tree").fit(Xg).kneighbors(Xg)
    ax.plot(np.sort(d[:, -1]) * KM_TIERRA, label=f"k = {k}")
ax.set(ylim=(0, 3), xlabel="Casos ordenados", ylabel="Distancia al k-ésimo vecino (km)", title="Gráfico k-distance (Guayas)")
ax.axhline(.35, color="k", ls=":"); ax.legend(); fig.tight_layout(); fig.savefig("figuras/fig_kdistance.png"); plt.show()

Guayas: 17,655 casos | coordenadas únicas: 12,819


In [12]:
filas = []
for mp in (15, 30, 50):
    for eps in (0.25, 0.35, 0.5, 1.0, 1.5, 2.0):
        lab = DBSCAN(eps=eps/KM_TIERRA, min_samples=mp, metric="haversine", algorithm="ball_tree").fit(Xg).labels_
        m = lab != -1; s = pd.Series(lab[m]).value_counts()
        sil = silhouette_score(Xg[m], lab[m], metric="haversine", sample_size=min(5000, m.sum()), random_state=SEMILLA) if len(s) > 1 else np.nan
        filas.append(dict(MinPts=mp, eps_km=eps, clusters=len(s), pct_en_cluster=round(m.mean()*100,1), pct_cluster_mayor=round(s.iloc[0]/len(g)*100,1), silueta=round(sil,3)))
grid = pd.DataFrame(filas); mostrar(grid); grid.to_csv("tablas/dbscan_sensibilidad.csv", index=False)
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
for mp, gg in grid.groupby("MinPts"):
    axs[0].plot(gg.eps_km, gg.clusters, marker="o", label=f"MinPts={mp}"); axs[1].plot(gg.eps_km, gg["pct_cluster_mayor"], marker="o", label=f"MinPts={mp}")
axs[0].set(title="Número de clústeres", xlabel="ε (km)"); axs[1].set(title="% de casos en el clúster mayor", xlabel="ε (km)")
axs[0].legend(); fig.tight_layout(); fig.savefig("figuras/fig_dbscan_sensibilidad.png"); plt.show()
RES["dbscan_original_1.5_15"] = grid[(grid.eps_km==1.5)&(grid.MinPts==15)].iloc[0].to_dict()

,MinPts,eps_km,clusters,pct_en_cluster,pct_cluster_mayor,silueta
0,15,0.25,118,72.5,16.1,0.285
1,15,0.35,60,85.1,32.1,0.113
2,15,0.50,42,90.4,58.7,-0.007
3,15,1.00,45,94.6,68.3,-0.036
4,15,1.50,40,95.6,69.1,0.183
5,15,2.00,33,96.4,77.9,0.482
6,30,0.25,83,48.6,2.3,0.623
7,30,0.35,64,71.3,15.8,0.272
8,30,0.50,31,85.3,32.4,0.349
9,30,1.00,27,91.5,67.6,0.112


**Decisión de parámetros.** Con ε = 1,5 km el clúster mayor concentra casi 7 de cada 10 casos: es toda el área metropolitana, no una geocerca. Se adopta **ε = 0,35 km** (mediana de la distancia al 30.º vecino, rodilla del k-distance) y **MinPts = 30**: zonas de escala de barrio, ninguna dominante.

In [13]:
EPS_KM, MINPTS = 0.35, 30
g["cl"] = DBSCAN(eps=EPS_KM/KM_TIERRA, min_samples=MINPTS, metric="haversine", algorithm="ball_tree").fit(Xg).labels_
m = g.cl != -1; tam = g.loc[m, "cl"].value_counts()
print(f"ε={EPS_KM} km, MinPts={MINPTS}: {len(tam)} zonas | {m.mean()*100:.1f}% de casos en zonas | ruido {100-m.mean()*100:.1f}% | zona mayor {tam.iloc[0]/len(g)*100:.1f}%")
RES["dbscan_final"] = dict(eps_km=EPS_KM, minpts=MINPTS, zonas=int(len(tam)), pct_en_zonas=round(m.mean()*100,1), pct_ruido=round(100-m.mean()*100,1), pct_zona_mayor=round(tam.iloc[0]/len(g)*100,1))

ε=0.35 km, MinPts=30: 64 zonas | 71.3% de casos en zonas | ruido 28.7% | zona mayor 15.8%


### 4.2 Densidad no es riesgo: severidad **estandarizada por año**
La proporción de desenlace adverso (fallecido o aún desaparecido) en Guayas pasa de ≈ 2 % (2017) a ≈ 26 % (2025). Comparar una zona con la tasa global la penaliza o favorece según los años en que tenga casos. Se usa **estandarización indirecta**: la tasa esperada de cada zona es el promedio de las tasas anuales de la provincia ponderado por los casos de la zona en cada año. Se contrasta con una prueba binomial unilateral y corrección de Benjamini-Hochberg (FDR).

In [14]:
g["adverso"] = g.situacion_actual.isin(["FALLECIDO","DESAPARECIDO"]).astype(int)
tasa_anual = g.groupby("anio").adverso.mean(); mostrar(tasa_anual.round(3).to_frame("tasa adversa Guayas"))

def severidad_zonas(d, col_cl, tasa_ref):
    filas = []
    for c, s in d[d[col_cl] != -1].groupby(col_cl):
        n, k = len(s), int(s.adverso.sum()); esp = float(s.anio.map(tasa_ref).mean())
        ya = s.groupby("anio").size().reindex(range(2017, 2026), fill_value=0)
        filas.append(dict(zona=c, casos=n, adversos=k, tasa=k/n, esperada=esp, razon_vs_esperada=(k/n)/esp, p=binomtest(k, n, esp, alternative="greater").pvalue,
                          anios_activos=int((ya >= 3).sum()), lat=s.lat.mean(), lon=s.lon.mean()))
    z = pd.DataFrame(filas); z["p_BH"] = multipletests(z.p, method="fdr_bh")[1] if len(z) else []
    z["severa"] = z.p_BH < 0.05; z["cronica"] = z.anios_activos >= 8
    return z.sort_values("p")

zonas = severidad_zonas(g, "cl", tasa_anual); mostrar(zonas.head(8).round(4)); zonas.to_csv("tablas/dbscan_zonas.csv", index=False)
print(f"Zonas: {len(zonas)} | severidad elevada (FDR<0,05, estandarizada por año): {int(zonas.severa.sum())} | crónicas (≥ 3 casos en ≥ 8 de 9 años): {int(zonas.cronica.sum())} ({zonas[zonas.cronica].casos.sum():,} casos)")
RES["zonas"] = dict(total=int(len(zonas)), severas=int(zonas.severa.sum()), cronicas=int(zonas.cronica.sum()),
                    severas_detalle=zonas[zonas.severa][["zona","casos","adversos","tasa","esperada","razon_vs_esperada","p_BH","anios_activos"]].round(4).to_dict("records"))
g = g.merge(zonas[["zona","severa"]].rename(columns={"zona":"cl"}), on="cl", how="left"); g["severa"] = g.severa.fillna(False).astype(bool)

,tasa adversa Guayas
anio,
2017,0.019
2018,0.019
2019,0.022
2020,0.038
2021,0.056
2022,0.110
2023,0.168
2024,0.229
2025,0.265


,zona,casos,adversos,tasa,esperada,razon_vs_esperada,p,anios_activos,lat,lon,p_BH,severa,cronica
44,44,55,17,0.3091,0.1112,2.7804,0.0001,7,-2.1264,-80.0158,0.0026,True,False
36,36,348,57,0.1638,0.0977,1.6772,0.0001,9,-2.1221,-79.9998,0.0026,True,True
46,46,50,15,0.3000,0.1124,2.6700,0.0003,7,-2.2477,-79.6424,0.0058,True,False
24,24,149,23,0.1544,0.0792,1.9484,0.0016,9,-2.3297,-79.4043,0.0253,True,True
1,1,86,15,0.1744,0.1019,1.7114,0.0268,9,-2.6287,-80.3890,0.3426,False,True
14,14,55,9,0.1636,0.0849,1.9285,0.0409,8,-2.7106,-80.2509,0.4362,False,True
39,39,56,12,0.2143,0.1360,1.5761,0.0709,7,-2.6717,-79.6218,0.5676,False,False
45,45,108,16,0.1481,0.0999,1.4830,0.0710,9,-2.0970,-79.6916,0.5676,False,True


Zonas: 64 | severidad elevada (FDR<0,05, estandarizada por año): 4 | crónicas (≥ 3 casos en ≥ 8 de 9 años): 37 (11,507 casos)


In [15]:
fig, axs = plt.subplots(1, 2, figsize=(14, 6.2))
ruido = g[g.cl == -1]; cz = g[g.cl != -1]
for ax, titulo in zip(axs, ["Zonas detectadas (ε=0,35 km; MinPts=30)", "Zonas con severidad elevada (FDR<0,05, estand. por año)"]):
    ax.scatter(ruido.lon, ruido.lat, s=3, c="lightgrey", label="Ruido"); ax.set(title=titulo, xlabel="Longitud", ylabel="Latitud")
axs[0].scatter(cz.lon, cz.lat, s=5, c=cz.cl, cmap="tab20")
axs[1].scatter(cz.lon, cz.lat, s=3, c="#c7d3e0", label="Otras zonas")
sv = g[g.severa]; axs[1].scatter(sv.lon, sv.lat, s=10, c=sv.cl, cmap="Set1", label="Severidad elevada")
for z in zonas[zonas.severa].itertuples(): axs[1].annotate(f"Z{z.zona}", (z.lon, z.lat), fontsize=9, weight="bold")
axs[1].legend(loc="lower left"); fig.tight_layout(); fig.savefig("figuras/fig_dbscan_mapa.png"); plt.show()

### 4.3 Validación temporal: ¿las zonas de 2017-2022 predicen 2023-2025?

In [16]:
ent, sal = g[g.anio <= 2022].copy().reset_index(drop=True), g[g.anio >= 2023].copy().reset_index(drop=True)
ent["cl"] = DBSCAN(eps=EPS_KM/KM_TIERRA, min_samples=MINPTS, metric="haversine", algorithm="ball_tree").fit(np.radians(ent[["lat","lon"]].to_numpy())).labels_
z_ent = severidad_zonas(ent, "cl", ent.groupby("anio").adverso.mean())
sev_ent = set(z_ent[z_ent.severa].zona); nom_ent = set(z_ent[z_ent.p < 0.05].zona)

core = ent[ent.cl != -1]; arbol = BallTree(np.radians(core[["lat","lon"]].to_numpy()), metric="haversine")
dist, idx = arbol.query(np.radians(sal[["lat","lon"]].to_numpy()), k=1)
dentro = dist[:, 0] * KM_TIERRA <= EPS_KM
sal["cl_ent"] = np.where(dentro, core.cl.to_numpy()[idx[:, 0]], -1)
tasa_sal = sal.groupby("anio").adverso.mean()
def evaluar(S, nombre):
    m = sal.cl_ent.isin(list(S)); n = int(m.sum())
    if n < 20: return dict(criterio=nombre, zonas=len(S), n_prueba=n, observados=np.nan, esperados=np.nan, razon=np.nan, p=np.nan)
    k = int(sal.loc[m, "adverso"].sum()); esp = float(sal.loc[m, "anio"].map(tasa_sal).mean())
    return dict(criterio=nombre, zonas=len(S), n_prueba=n, observados=k, esperados=round(esp*n,1), razon=round(k/(esp*n),2), p=binomtest(k, n, esp, alternative="greater").pvalue)
val_z = pd.DataFrame([evaluar(sev_ent, "Severas en 2017-2022 (FDR<0,05)"), evaluar(nom_ent, "Nominal p<0,05 en 2017-2022 (exploratorio)")])
print(f"Zonas de entrenamiento: {len(z_ent)} | cobertura: {dentro.mean()*100:.1f}% de los casos 2023-2025 caen en zonas detectadas con datos 2017-2022")
mostrar(val_z); val_z.to_csv("tablas/dbscan_validacion_temporal.csv", index=False)
RES["validacion_temporal_zonas"] = dict(zonas_entrenamiento=int(len(z_ent)), cobertura_pct=round(dentro.mean()*100,1), resultados=val_z.replace({np.nan: None}).to_dict("records"))

Zonas de entrenamiento: 46 | cobertura: 62.6% de los casos 2023-2025 caen en zonas detectadas con datos 2017-2022


,criterio,zonas,n_prueba,observados,esperados,razon,p
0,"Severas en 2017-2022 (FDR<0,05)",0,0,NaN,NaN,NaN,NaN
1,"Nominal p<0,05 en 2017-2022 (exploratorio)",1,40,13.0,8.9,1.45,0.092062


## 5. Minería de reglas de asociación (Apriori)
### 5.1 Jerarquía entre las variables `motivo` y `estado`
`EXTRAVIADO - DISCAPACIDAD…` y `EXTRAVIADO - AUSENCIA TEMPORAL` (variable `motivo`) son *exactamente* los casos `EXTRAVÍO` (variable `estado`): 4 216 + 2 104 = 6 320. Una regla entre ambos tiene confianza 1,0 por construcción y su Lift (≈ 12) mide la taxonomía, no un patrón de riesgo. Además `FALLECIDO`/`NO_APLICA` en `motivo` y `EN INVESTIGACIÓN` en `estado` ya codifican el desenlace.

### 5.2 Diseño de las transacciones
* **Antecedentes** (perfil): sexo, grupo de edad, etnia, nacionalidad, región y motivo (sin categorías que codifican el desenlace; `EXTRAVIADO-*` fusionadas).
* **Consecuentes** (desenlace): `FALLECIDO`, `SIGUE_DESAPARECIDO`, `RESOLUCION>30d`.
* Soporte mínimo 0,1 %; se conservan reglas con ≥ 50 casos, **límite inferior del IC95 % de la confianza** que implique Lift > 1,5, y sin redundancia respecto de reglas más generales.

In [17]:
def construir_transacciones(d):
    mot = d.motivo_desaparicion.where(~d.motivo_desaparicion.str.startswith("EXTRAVIADO"), "EXTRAVIADO")
    mot = mot.where(~mot.isin(["FALLECIDO","NO_APLICA","CERRADO POR FISCALÍA / DELITO REFORMULADO","DISCRIMINACIÓN"]), np.nan)
    et  = d.etnia.where(d.etnia.isin(["MESTIZO/A","INDIGENA","AFRO","MONTUBIO/A"]), "OTRA_ETNIA").where(d.etnia != "SIN_DATO", np.nan)
    nac = pd.Series(np.where(d.nacionalidad=="ECUADOR","ECUATORIANA","EXTRANJERA"), index=d.index).where(d.nacionalidad != "SIN_DATO", np.nan)
    perfil = pd.DataFrame({"sexo": d.sexo, "edad": d.edad_g, "etnia": et, "nac": nac, "region": d.region, "motivo": mot})
    items = pd.get_dummies(perfil, prefix_sep="=").astype(bool)
    sal_ = pd.DataFrame({"SALE=FALLECIDO": d.situacion_actual=="FALLECIDO", "SALE=SIGUE_DESAPARECIDO": d.situacion_actual=="DESAPARECIDO", "SALE=RESOLUCION>30d": d.t > 30})
    return pd.concat([items, sal_], axis=1), list(sal_.columns)

def minar_reglas(T, cols_salida, min_support=0.001, n_min=50, lift_min=1.5):
    N = len(T); fi = apriori(T, min_support=min_support, max_len=4, use_colnames=True)
    R = association_rules(fi, metric="lift", min_threshold=lift_min)
    R = R[R.apply(lambda r: len(r.consequents)==1 and list(r.consequents)[0] in cols_salida and not (set(r.antecedents) & set(cols_salida)), axis=1)].copy()
    R["n_regla"] = (R.support*N).round().astype(int); R["n_ant"] = (R["antecedent support"]*N).round().astype(int)
    R["conf_lo"], R["conf_hi"] = proportion_confint(R.n_regla, R.n_ant, method="wilson")
    R["ant"] = R.antecedents.apply(lambda s: " & ".join(sorted(s))); R["con"] = R.consequents.apply(lambda s: list(s)[0]); R["len_ant"] = R.antecedents.apply(len)
    R = R[(R.n_regla >= n_min) & (R.conf_lo / R["consequent support"] > lift_min)]
    keep = []
    for _, r in R.iterrows():
        gen = R[(R.con==r.con) & (R.len_ant < r.len_ant) & R.antecedents.apply(lambda s: s < r.antecedents)]
        keep.append(not (gen.lift >= 0.9*r.lift).any())
    return R[keep].sort_values(["con","lift"], ascending=[True, False]), fi

T, cols_salida = construir_transacciones(df)
print(f"Transacciones: {len(T):,} | ítems: {T.shape[1]}"); mostrar(T[cols_salida].mean().round(4).to_frame("prevalencia"))
reglas, fi = minar_reglas(T, cols_salida)
print(f"Conjuntos frecuentes: {len(fi):,} | reglas finales: {len(reglas)}")
cols = ["ant","con","n_ant","n_regla","confidence","conf_lo","conf_hi","lift"]
for c in cols_salida:
    print(f"\n### {c}  (prevalencia {T[c].mean()*100:.1f}%)"); mostrar(reglas[reglas.con==c][cols].head(8).round(3))
reglas[["ant","con","n_ant","n_regla","support","confidence","conf_lo","conf_hi","lift"]].round(4).to_csv("tablas/reglas_apriori.csv", index=False)
RES["apriori"] = dict(n_reglas=int(len(reglas)), top=reglas.groupby("con").head(3)[["ant","con","n_ant","confidence","lift"]].round(3).to_dict("records"))

Transacciones: 75,748 | ítems: 28


,prevalencia
SALE=FALLECIDO,0.0344
SALE=SIGUE_DESAPARECIDO,0.0334
SALE=RESOLUCION>30d,0.2174


Conjuntos frecuentes: 2,449 | reglas finales: 48

### SALE=FALLECIDO  (prevalencia 3.4%)


,ant,con,n_ant,n_regla,confidence,conf_lo,conf_hi,lift
1033,edad=ADULTO & etnia=AFRO & sexo=HOMBRE,SALE=FALLECIDO,645,98,0.152,0.126,0.182,4.421
3536,edad=ADULTO & etnia=AFRO & region=COSTA,SALE=FALLECIDO,603,85,0.141,0.115,0.171,4.102
2111,etnia=AFRO & region=COSTA & sexo=HOMBRE,SALE=FALLECIDO,687,96,0.140,0.116,0.168,4.066
1466,edad=ADULTO & region=COSTA & sexo=HOMBRE,SALE=FALLECIDO,8108,1062,0.131,0.124,0.139,3.812
1394,edad=ADULTO & region=AMAZONIA & sexo=HOMBRE,SALE=FALLECIDO,736,90,0.122,0.101,0.148,3.558
26,edad=ADULTO MAYOR,SALE=FALLECIDO,2984,354,0.119,0.108,0.131,3.452
457,edad=ADULTO & etnia=AFRO,SALE=FALLECIDO,885,104,0.118,0.098,0.140,3.420
198,etnia=AFRO & sexo=HOMBRE,SALE=FALLECIDO,1036,116,0.112,0.094,0.133,3.258



### SALE=SIGUE_DESAPARECIDO  (prevalencia 3.3%)


,ant,con,n_ant,n_regla,confidence,conf_lo,conf_hi,lift
3550,edad=ADULTO & etnia=AFRO & region=COSTA,SALE=SIGUE_DESAPARECIDO,603,137,0.227,0.196,0.262,6.802
1047,edad=ADULTO & etnia=AFRO & sexo=HOMBRE,SALE=SIGUE_DESAPARECIDO,645,146,0.226,0.196,0.260,6.777
2125,etnia=AFRO & region=COSTA & sexo=HOMBRE,SALE=SIGUE_DESAPARECIDO,687,145,0.211,0.182,0.243,6.319
2638,nac=EXTRANJERA & region=COSTA & sexo=HOMBRE,SALE=SIGUE_DESAPARECIDO,443,81,0.183,0.150,0.222,5.474
463,edad=ADULTO & etnia=AFRO,SALE=SIGUE_DESAPARECIDO,885,161,0.182,0.158,0.209,5.447
1363,edad=ADULTO & nac=EXTRANJERA & sexo=HOMBRE,SALE=SIGUE_DESAPARECIDO,706,126,0.178,0.152,0.208,5.343
4042,edad=ADULTO & nac=EXTRANJERA & region=COSTA,SALE=SIGUE_DESAPARECIDO,498,86,0.173,0.142,0.208,5.170
204,etnia=AFRO & sexo=HOMBRE,SALE=SIGUE_DESAPARECIDO,1036,171,0.165,0.144,0.189,4.942



### SALE=RESOLUCION>30d  (prevalencia 21.7%)


,ant,con,n_ant,n_regla,confidence,conf_lo,conf_hi,lift
1378,edad=ADULTO & nac=EXTRANJERA & sexo=HOMBRE,SALE=RESOLUCION>30d,706,353,0.500,0.463,0.537,2.300
551,edad=ADULTO & nac=EXTRANJERA,SALE=RESOLUCION>30d,1162,520,0.448,0.419,0.476,2.058
469,edad=ADULTO & etnia=AFRO,SALE=RESOLUCION>30d,885,382,0.432,0.399,0.464,1.985
293,nac=EXTRANJERA & sexo=HOMBRE,SALE=RESOLUCION>30d,1096,469,0.428,0.399,0.457,1.968
211,etnia=AFRO & sexo=HOMBRE,SALE=RESOLUCION>30d,1036,399,0.385,0.356,0.415,1.771
51,nac=EXTRANJERA,SALE=RESOLUCION>30d,2356,900,0.382,0.363,0.402,1.757


### 5.3 Validación temporal de las reglas (minadas en 2017-2022, evaluadas en 2023-2025)

In [18]:
dtr, dte = df[df.anio <= 2022], df[df.anio >= 2023]
Ttr, _ = construir_transacciones(dtr); Tte, _ = construir_transacciones(dte)
Tte = Tte.reindex(columns=Ttr.columns, fill_value=False)
r_tr, _ = minar_reglas(Ttr, cols_salida)
filas = []
for r in r_tr.itertuples():
    ant = list(r.antecedents); m = Tte[ant].all(axis=1); n = int(m.sum())
    conf = Tte.loc[m, r.con].mean() if n else np.nan; base = Tte[r.con].mean()
    filas.append(dict(regla=f"{r.ant} → {r.con}", lift_entrenamiento=round(r.lift,2), n_prueba=n, confianza_prueba=round(conf,3), lift_prueba=round(conf/base,2) if n else np.nan))
val = pd.DataFrame(filas)
if len(val):
    mostrar(val.head(12))
    persist = (val.lift_prueba > 1.5).mean()
    print(f"Reglas minadas en 2017-2022: {len(val)} | con Lift>1,5 en 2023-2025: {int((val.lift_prueba>1.5).sum())} ({persist*100:.0f}%) | Lift>1: {(val.lift_prueba>1).mean()*100:.0f}%")
    val.to_csv("tablas/reglas_validacion_temporal.csv", index=False)
    RES["apriori_validacion"] = dict(n=int(len(val)), pct_lift_gt_1_5=round(persist*100,1), pct_lift_gt_1=round((val.lift_prueba>1).mean()*100,1))
    por_con = val.assign(con=val.regla.str.split(" → ").str[1]).groupby("con").lift_prueba.agg(["count", "median", lambda s: (s>1.5).mean()]).round(2)
    por_con.columns = ["reglas", "lift_mediano_prueba", "prop_lift>1,5"]; mostrar(por_con)
    RES["apriori_validacion_por_consecuente"] = por_con.reset_index().to_dict("records")
else:
    print("Ninguna regla supera los filtros en el periodo de entrenamiento.")

,regla,lift_entrenamiento,n_prueba,confianza_prueba,lift_prueba
0,edad=ADULTO MAYOR & sexo=HOMBRE → SALE=FALLECIDO,5.15,788,0.148,2.49
1,edad=ADULTO MAYOR → SALE=FALLECIDO,4.58,1017,0.139,2.32
2,edad=ADULTO & sexo=HOMBRE → SALE=FALLECIDO,3.21,6288,0.156,2.61
3,sexo=HOMBRE → SALE=FALLECIDO,2.36,9808,0.121,2.02
4,edad=ADULTO → SALE=FALLECIDO,2.00,9655,0.113,1.90
5,motivo=CAUSAS PERSONALES → SALE=RESOLUCION>30d,5.32,7055,0.221,0.77
6,edad=ADULTO & nac=EXTRANJERA & sexo=HOMBRE → S...,2.35,263,0.597,2.08
7,edad=ADULTO & nac=EXTRANJERA → SALE=RESOLUCION...,2.09,380,0.561,1.95
8,edad=ADULTO & etnia=AFRO → SALE=RESOLUCION>30d,1.89,344,0.552,1.92
9,nac=EXTRANJERA → SALE=RESOLUCION>30d,1.83,800,0.455,1.59


Reglas minadas en 2017-2022: 26 | con Lift>1,5 en 2023-2025: 22 (85%) | Lift>1: 96%


,reglas,lift_mediano_prueba,"prop_lift>1,5"
con,,,
SALE=FALLECIDO,5,2.32,1.00
SALE=RESOLUCION>30d,6,1.76,0.67
SALE=SIGUE_DESAPARECIDO,15,2.51,0.87


## 6. Integración de los tres motores (Cox fuera de muestra, estratificado por año)
Para Guayas 2023-2025 se estima el efecto sobre la **velocidad de cierre** de (i) estar en una zona de densidad detectada con datos 2017-2022 (DBSCAN), (ii) tener un perfil de riesgo (etnia o nacionalidad, Lift ≥ 2) señalado por las reglas minadas en 2017-2022 (Apriori), y (iii) sexo y edad. HR < 1 = cierre más lento. Zonas y perfiles se definen con datos anteriores para evitar circularidad, y el modelo se **estratifica por año** para no confundir con la tendencia temporal.

In [19]:
# Perfil de riesgo = reglas minadas en 2017-2022 cuyo antecedente incluye etnia o nacionalidad (sexo y edad entran como covariables
# propias del Cox; una unión de perfiles demográficos amplios marcaría a ~79 % de los casos y se solaparía con ellas) y Lift de entrenamiento >= 2
cand = r_tr[r_tr.con.isin(["SALE=SIGUE_DESAPARECIDO","SALE=RESOLUCION>30d"])]
sel  = cand[cand.antecedents.apply(lambda s: any(i.startswith(("etnia=","nac=")) for i in s)) & (cand.lift >= 2)]
ants = [set(a) for a in sel.antecedents]
perfiles = sorted({" & ".join(sorted(a)) for a in ants}); print("Perfiles de riesgo (entrenamiento):", perfiles)
Tg, _ = construir_transacciones(df); Tg = Tg.reindex(columns=Ttr.columns, fill_value=False)
perfil = np.zeros(len(df), dtype=bool)
for a in ants: perfil |= Tg[list(a)].all(axis=1).to_numpy()
df["perfil_riesgo"] = perfil.astype(int)

gt = df.loc[(df.provincia=="GUAYAS") & (df.anio >= 2023) & df.edad_g.notna()].copy()
dist, idx = arbol.query(np.radians(gt[["lat","lon"]].to_numpy()), k=1)
gt["en_zona_densa"] = (dist[:,0]*KM_TIERRA <= EPS_KM).astype(int)
gt["en_zona_severa"] = (np.isin(core.cl.to_numpy()[idx[:,0]], list(sev_ent)) & (dist[:,0]*KM_TIERRA <= EPS_KM)).astype(int)
gt["mujer"] = (gt.sexo=="MUJER").astype(int)
for g_ in ["NIÑOS(AS)","ADULTO","ADULTO MAYOR"]: gt[g_] = (gt.edad_g==g_).astype(int)
print(f"Prevalencia del perfil de riesgo en Guayas 2023-2025: {gt.perfil_riesgo.mean()*100:.1f}%")
print(f"Casos Guayas 2023-2025: {len(gt):,} | en zona densa: {int(gt.en_zona_densa.sum())} | en zona severa: {int(gt.en_zona_severa.sum())} | con perfil de riesgo: {int(gt.perfil_riesgo.sum())}")

covs = ["en_zona_densa","perfil_riesgo","mujer","NIÑOS(AS)","ADULTO","ADULTO MAYOR"]
if gt.en_zona_severa.sum() >= 20: covs.insert(1, "en_zona_severa")
mod = CoxPHFitter().fit(gt[["t","evento","anio"] + covs], "t", "evento", strata=["anio"])
tab_cox = mod.summary[["exp(coef)","exp(coef) lower 95%","exp(coef) upper 95%","p"]].round(3); mostrar(tab_cox); tab_cox.to_csv("tablas/cox_integrado.csv")
RES["cox_integrado"] = tab_cox.reset_index().to_dict("records")
RES["perfiles_riesgo"] = perfiles

Perfiles de riesgo (entrenamiento): ['edad=ADULTO & etnia=AFRO', 'edad=ADULTO & nac=EXTRANJERA', 'edad=ADULTO & nac=EXTRANJERA & sexo=HOMBRE', 'etnia=AFRO', 'etnia=AFRO & sexo=HOMBRE', 'nac=EXTRANJERA', 'nac=EXTRANJERA & sexo=HOMBRE']


Prevalencia del perfil de riesgo en Guayas 2023-2025: 5.9%
Casos Guayas 2023-2025: 4,872 | en zona densa: 3048 | en zona severa: 0 | con perfil de riesgo: 288


,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,p
covariate,,,,
en_zona_densa,1.091,1.025,1.162,0.006
perfil_riesgo,0.701,0.613,0.803,0.000
mujer,1.233,1.148,1.324,0.000
NIÑOS(AS),0.933,0.808,1.076,0.339
ADULTO,0.878,0.815,0.945,0.001
ADULTO MAYOR,1.056,0.911,1.225,0.467


## 7. Cifras clave para la memoria

In [20]:
def limpiar(o):
    if isinstance(o, dict):  return {str(k): limpiar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [limpiar(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    return o
json.dump(limpiar(RES), open("tablas/resultados_clave.json", "w"), ensure_ascii=False, indent=2)
print("Resultados guardados en tablas/resultados_clave.json")

Resultados guardados en tablas/resultados_clave.json
